In [1]:
import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(42)

# ── 1. Numerical Jacobian — the universal gradient checker ─
def numerical_jacobian(f, x, eps=1e-5):
    """Compute full Jacobian of f at x numerically."""
    x = x.astype(np.float64)
    y0 = f(x)
    m = len(y0)   # output dim
    n = len(x)    # input dim
    J = np.zeros((m, n))
    for j in range(n):
        x_plus  = x.copy(); x_plus[j]  += eps
        x_minus = x.copy(); x_minus[j] -= eps
        J[:, j] = (f(x_plus) - f(x_minus)) / (2 * eps)
    return J

# ── 2. Jacobian of a linear layer ─────────────────────────
W = np.random.randn(3, 4)   # (3 outputs, 4 inputs)
f_linear = lambda x: W @ x

x = np.random.randn(4)
J_numerical  = numerical_jacobian(f_linear, x)
J_analytical = W                 # Jacobian of Wx IS W

print("Linear layer Jacobian:")
print(f"Numerical:{J_numerical.round(4)}")
print(f"Analytical (W):{J_analytical.round(4)}")
print(f"Match: {np.allclose(J_numerical, J_analytical, atol=1e-6)}")

# ── 3. Softmax Jacobian ────────────────────────────────────
def softmax(x):
    e = np.exp(x - x.max())   # numerically stable
    return e / e.sum()

def softmax_jacobian_analytical(x):
    y = softmax(x)
    return np.diag(y) - np.outer(y, y)   # diag(y) - y@yᵀ

x_soft = np.array([1.0, 2.0, 0.5, -1.0])
J_soft_num = numerical_jacobian(softmax, x_soft)
J_soft_ana = softmax_jacobian_analytical(x_soft)

print("Softmax Jacobian shape:", J_soft_ana.shape)  # (4,4) — dense
print(f"Numerical vs analytical match: {np.allclose(J_soft_num, J_soft_ana, atol=1e-6)}")
print("Jacobian (shows off-diagonal terms — dense unlike ReLU):")
print(J_soft_ana.round(4))

# ── 4. ReLU Jacobian — diagonal (contrast with softmax) ───
def relu(x):    return np.maximum(0, x)
x_relu = np.array([1.5, -0.3, 0.8, -1.2])
J_relu = numerical_jacobian(relu, x_relu)
print("ReLU Jacobian (diagonal — no cross-output dependencies):")
print(J_relu.round(3))
print("Diagonal only?", np.allclose(J_relu, np.diag(np.diag(J_relu))))

# ── 5. VJP — what PyTorch actually computes ───────────────
# Instead of full Jacobian, compute v @ J (vector-Jacobian product)
def vjp(f, x, v, eps=1e-5):
    """Compute vᵀ @ J without forming J explicitly."""
    J = numerical_jacobian(f, x, eps)
    return v @ J

v = np.array([1.0, -0.5, 0.3, 0.8])   # upstream gradient
vjp_result = vjp(softmax, x_soft, v)
print(f"VJP (upstream grad @ softmax Jacobian): {vjp_result.round(6)}")

# Verify with PyTorch autograd
x_t = torch.tensor(x_soft, requires_grad=True, dtype=torch.float64)
y_t = torch.softmax(x_t, dim=0)
y_t.backward(torch.tensor(v))
print(f"PyTorch VJP: {x_t.grad.numpy().round(6)}")
print(f"Match: {np.allclose(vjp_result, x_t.grad.numpy(), atol=1e-6)}")

# ── 6. Batch backprop — gradient shapes ───────────────────
print("=== BATCH BACKPROP ===")
B, n_in, n_out = 8, 4, 3   # batch=8, input_dim=4, output_dim=3

X = torch.randn(B, n_in,  requires_grad=True)
W = torch.randn(n_out, n_in, requires_grad=True)
b = torch.randn(n_out,       requires_grad=True)

# Forward: (B,4) → (B,3)
Z = X @ W.T + b     # (B, n_out)
A = torch.relu(Z)
L = A.mean()        # scalar loss

L.backward()

print(f"X shape: {X.shape},    dL/dX shape: {X.grad.shape}")     # (8,4)
print(f"W shape: {W.shape},    dL/dW shape: {W.grad.shape}")     # (3,4) ← same as W
print(f"b shape: {b.shape},    dL/db shape: {b.grad.shape}")     # (3,)
print(f"W gradient is average over batch: {W.grad}")
print("Save as day09_jacobians.ipynb and push to GitHub.")

Linear layer Jacobian:
Numerical:[[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]
 [-0.4695  0.5426 -0.4634 -0.4657]]
Analytical (W):[[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]
 [-0.4695  0.5426 -0.4634 -0.4657]]
Match: True
Softmax Jacobian shape: (4, 4)
Numerical vs analytical match: True
Jacobian (shows off-diagonal terms — dense unlike ReLU):
[[ 0.1739 -0.1366 -0.0305 -0.0068]
 [-0.1366  0.238  -0.0829 -0.0185]
 [-0.0305 -0.0829  0.1175 -0.0041]
 [-0.0068 -0.0185 -0.0041  0.0294]]
ReLU Jacobian (diagonal — no cross-output dependencies):
[[1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 0.]]
Diagonal only? True
VJP (upstream grad @ softmax Jacobian): [ 0.227672 -0.295313  0.042898  0.024743]
PyTorch VJP: [ 0.227672 -0.295313  0.042898  0.024743]
Match: True
=== BATCH BACKPROP ===
X shape: torch.Size([8, 4]),    dL/dX shape: torch.Size([8, 4])
W shape: torch.Size([3, 4]),    dL/dW shape: torch.Size([3, 4])
b shape: torch.Size([3]),